# 02 · Entrenamiento en Google Colab (GPU)

Mismo pipeline que en local, con MLflow guardado en Google Drive. Al terminar, los runs se copian al MLflow de la
aplicación con `potato_blight.sync_runs` (ver docs/guia_entrenamiento.md).

**Presupuesto idéntico para los cinco modelos** (`tuning` en `configs/base.yaml`): no lo cambie para un solo modelo.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/papasana"  # carpeta con data/ (imágenes y manifest) y mlflow.db

In [ ]:
!git clone https://github.com/<su-usuario>/<su-repositorio>.git repo
%cd repo/ml
!pip install -q -e .

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{BASE}/mlflow.db"
!ln -sfn {BASE}/data data
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Prueba de humo (imágenes sintéticas, 1 época): verifica el entorno antes de gastar horas de GPU
!python -m potato_blight.smoke --arch resnet50 --epochs 1

In [ ]:
ARQUITECTURAS = ["resnet50", "efficientnet_b3", "convnext_tiny", "cnn_vit", "effnet_att_bilstm"]
for arch in ARQUITECTURAS:
    # 1) Búsqueda con Optuna (mismo presupuesto para todos) → configs/tuned/<arch>.yaml
    !python -m potato_blight.tune --config configs/{arch}.yaml --storage sqlite:///{BASE}/optuna.db

In [ ]:
for arch in ARQUITECTURAS:
    # 2) Validación cruzada agrupada (5 particiones) con los hiperparámetros elegidos
    !python -m potato_blight.train --config configs/tuned/{arch}.yaml --mode cv

In [ ]:
# 3) Modelo final de cada arquitectura (se registra después en el MLflow de la aplicación)
for arch in ARQUITECTURAS:
    !python -m potato_blight.train --config configs/tuned/{arch}.yaml --mode final

In [ ]:
# 4) Comparación estadística (Friedman, Wilcoxon-Holm, McNemar, DeLong, bootstrap)
!python -m potato_blight.compare --metric macro_f1